# 🦙 Qwen 3.5 Tiny Models — Vision RAG Demo

**The plot:** Alibaba dropped the Qwen 3.5 Small series (0.8B → 9B) in early 2026 with **native multimodal vision built in** — not bolted on. These run on a laptop. Some run on a phone.

This notebook shows:
1. Running Qwen 3.5 locally via **Ollama** (free, private, offline)
2. Using the **DashScope API** for cloud access with vision
3. A **text + image RAG pipeline** — retrieve relevant content, answer with grounding
4. Why tiny models change the RAG game for edge/private deployments

---

## Model sizes available

| Model | Params | Vision | Runs on | RAM |
|-------|--------|--------|---------|-----|
| Qwen3.5-0.8B | 800M | ✅ native | Phone / IoT | ~1GB |
| Qwen3.5-2B | 2B | ✅ native | Laptop | ~2GB |
| Qwen3.5-4B | 4B | ✅ native | Desktop | ~4GB |
| Qwen3.5-9B | 9B | ✅ native | High-end PC | ~8GB |

Architecture: Gated DeltaNet + sparse MoE, 262k context window. The 9B rivals models 10x its size on benchmarks.

---

## Two ways to run: pick your path

```
PATH A: Local (Ollama)         PATH B: Cloud (DashScope API)
─────────────────────          ────────────────────────────
Free, private, offline         Bigger models, vision API
Any model size                 qwen3.5-plus (hosted)
Needs: ~4GB RAM                Needs: ALIKEY env var
Best for: production, privacy  Best for: prototyping, Colab
```

In [ ]:
# ─── 0. Setup — detect environment and install deps ───────────────────────
import os, sys, subprocess

IS_COLAB = 'google.colab' in sys.modules
print(f"Environment: {'Google Colab' if IS_COLAB else 'Local'}")

# Install Python deps
!pip install -q openai sentence-transformers faiss-cpu Pillow requests

print("✅ Python deps installed")

In [ ]:
# ─── 1. Choose your backend ───────────────────────────────────────────────
#
# Set BACKEND = "ollama" for local Qwen3.5 via Ollama
# Set BACKEND = "dashscope" for cloud API (needs ALIKEY)
#

BACKEND = "ollama"  # <── change this

# Ollama config
OLLAMA_MODEL = "qwen3.5:2b"   # options: qwen3.5:0.8b, qwen3.5:2b, qwen3.5:4b, qwen3.5:9b

# DashScope config  
DASHSCOPE_MODEL = "qwen3.5-plus"  # vision-capable cloud model
DASHSCOPE_API_KEY = os.environ.get("ALIKEY", "")  # set your key or paste here
DASHSCOPE_BASE_URL = "https://coding-intl.dashscope.aliyuncs.com/v1"

print(f"Backend: {BACKEND}")
if BACKEND == "dashscope" and not DASHSCOPE_API_KEY:
    print("⚠️  ALIKEY not set — get a free key at https://dashscope.aliyuncs.com")

## Part 1: Ollama Setup (skip if using DashScope)

In [ ]:
# ─── 1a. Install Ollama (Colab / Linux) ───────────────────────────────────
if BACKEND == "ollama":
    import shutil, time

    if not shutil.which("ollama"):
        print("Installing Ollama...")
        !curl -fsSL https://ollama.com/install.sh | sh
    else:
        print("Ollama already installed")

    # Start server in background
    proc = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    time.sleep(10)  # give it time to warm up
    print("Ollama server started")

    # Pull the model (first run only — ~1.5GB for 2B)
    print(f"\nPulling {OLLAMA_MODEL} (first run downloads model)...")
    result = subprocess.run(
        ["ollama", "pull", OLLAMA_MODEL],
        capture_output=True, text=True
    )
    print(result.stdout[-200:] if result.returncode == 0 else result.stderr[-200:])
    print(f"✅ {OLLAMA_MODEL} ready")

In [ ]:
# ─── 1b. Create a unified chat function for both backends ─────────────────
from openai import OpenAI

def get_client():
    if BACKEND == "ollama":
        return OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    else:  # dashscope
        return OpenAI(base_url=DASHSCOPE_BASE_URL, api_key=DASHSCOPE_API_KEY)

def get_model():
    return OLLAMA_MODEL if BACKEND == "ollama" else DASHSCOPE_MODEL

def chat(messages, stream=False):
    """Call the active model with a list of messages."""
    client = get_client()
    response = client.chat.completions.create(
        model=get_model(),
        messages=messages,
        stream=stream
    )
    if stream:
        full = ""
        for chunk in response:
            if chunk.choices[0].delta.content:
                token = chunk.choices[0].delta.content
                print(token, end="", flush=True)
                full += token
        print()
        return full
    return response.choices[0].message.content

# Quick smoke test
reply = chat([{"role": "user", "content": "Say exactly: QWEN3.5 IS ALIVE"}])
print(f"Model says: {reply}")

---
## Part 2: Build a RAG Knowledge Base

We'll build a mini knowledge base about AI topics — mixing text articles and image-described content. The RAG pipeline will retrieve relevant chunks before answering.

In [ ]:
# ─── 2. Our knowledge base — text documents ───────────────────────────────

documents = [
    {
        "id": "doc_01",
        "title": "Qwen 3.5 Small Models",
        "type": "text",
        "content": """The Qwen 3.5 Small series (0.8B to 9B parameters) was released by Alibaba in March 2026.
These models feature native multimodal architecture — text, images, and video are handled by the
same weights, not via adapters. The 0.8B model runs on phones and IoT devices. The 9B model
outperforms many 70B+ models on benchmarks. All sizes use a Gated DeltaNet hybrid attention
system with up to 262,000 token context windows. Models are available open-source on Hugging Face
under Apache-2.0 and via Ollama as qwen3.5:0.8b, qwen3.5:2b, etc."""
    },
    {
        "id": "doc_02",
        "title": "RAG Architecture Patterns",
        "type": "text",
        "content": """Retrieval-Augmented Generation (RAG) combines a retrieval system with a language model.
The classic pipeline: (1) embed documents into vectors, (2) store in a vector database like FAISS,
(3) at query time, embed the question, (4) retrieve top-K similar documents, (5) inject them as
context into the LLM prompt, (6) generate a grounded answer. Modern RAG variants include:
HyDE (generate a hypothetical answer first, use it for retrieval), multi-hop RAG (chain multiple
retrievals), and graph RAG (traverse a knowledge graph). The key metric is faithfulness —
does the answer actually come from the retrieved context?"""
    },
    {
        "id": "doc_03",
        "title": "Vision RAG — Multimodal Retrieval",
        "type": "text",
        "content": """Vision RAG extends classic RAG to handle image inputs. There are two approaches:
(A) Caption-first: use a vision model to caption all images, embed the captions as text, retrieve
captions, then optionally pass the original image to the LLM. (B) Native multimodal: embed both
images and text in a shared embedding space (e.g. CLIP, or a vision-language model), retrieve
mixed image+text results, pass everything to a multimodal LLM. Qwen 3.5 enables approach B
cheaply since it runs locally and handles image inputs natively. Vision RAG is powerful for
document understanding, product catalogs, medical imaging, and diagram Q&A."""
    },
    {
        "id": "doc_04",
        "title": "Hallucination and Grounding",
        "type": "text",
        "content": """LLM hallucination happens when the model generates plausible-sounding but factually wrong
information. RAG reduces hallucination by injecting retrieved evidence into the prompt, giving
the model something concrete to anchor to. However RAG doesn't eliminate hallucination — the
model can still ignore the context, misread it, or confabulate when the context doesn't contain
the answer. Mitigation strategies: (1) cite-forcing (instruct the model to quote the source),
(2) faithfulness scoring (check if the answer claims are entailed by context), (3) hybrid search
(keyword + semantic to avoid retrieval failures), (4) confabulation detection with a judge LLM."""
    },
    {
        "id": "doc_05",
        "title": "Tiny Models vs. Large Models for RAG",
        "type": "text",
        "content": """For RAG tasks, tiny models (2B-9B) are often competitive with large models because:
1) RAG reduces the knowledge demand on the model — answers come from retrieved context, not
memorized facts. 2) The model mainly needs to follow instructions and summarize — which small
models do well. 3) Inference is 10-50x faster, enabling real-time applications. 4) Local
deployment means zero data leaves the device — crucial for healthcare, legal, and personal data.
Benchmarks on RAG-specific evals (RAGAS, ARES) show Qwen 3.5 9B matching GPT-4-mini on most
RAG tasks while running entirely offline."""
    },
    {
        "id": "doc_06",
        "title": "FAISS Vector Database",
        "type": "text",
        "content": """FAISS (Facebook AI Similarity Search) is a library for efficient similarity search over
dense vectors. It supports exact search (IndexFlatL2) and approximate search (IndexIVFFlat,
IndexHNSW) for billion-scale indexes. For local RAG with thousands of documents, IndexFlatL2
is fast enough and exact — no approximation errors. FAISS runs entirely in-memory and in-process,
making it ideal for edge RAG deployments with tiny models. The typical embedding dimension is
384 (MiniLM) or 768 (MPNet). FAISS stores raw vectors — you maintain a parallel list mapping
vector indices to document IDs."""
    },
    {
        "id": "doc_07",
        "title": "On-Device AI Privacy Benefits",
        "type": "text",
        "content": """Running AI models on-device (phone, laptop, edge server) provides strong privacy guarantees:
user data never leaves the device, no cloud API calls, no logging by third parties. This matters
for healthcare (patient records), legal (privileged documents), personal assistants (private notes),
and regulated industries. Tiny models like Qwen 3.5 0.8B/2B make fully private AI assistants
feasible even on consumer hardware. The tradeoff: lower quality than cloud models for complex
reasoning, but for RAG the quality gap narrows significantly since answers come from retrieved
context rather than model knowledge."""
    },
    {
        "id": "doc_08",
        "title": "Sentence Transformers for Embeddings",
        "type": "text",
        "content": """Sentence Transformers is a Python library for generating text embeddings using pre-trained
transformer models. The most common model for RAG is all-MiniLM-L6-v2 (22M params, 384-dim,
fast) or all-mpnet-base-v2 (better quality, 768-dim). For multilingual use: paraphrase-multilingual-
MiniLM-L12-v2 covers 50+ languages. Usage: `from sentence_transformers import SentenceTransformer;
model = SentenceTransformer('all-MiniLM-L6-v2'); embeddings = model.encode(texts)`. These run
locally, are free, and combined with Qwen 3.5 and FAISS give you a completely offline RAG stack."""
    },
]

print(f"Knowledge base: {len(documents)} documents loaded")
for d in documents:
    print(f"  [{d['id']}] {d['title']}")

In [ ]:
# ─── 3. Build the vector index with Sentence Transformers + FAISS ─────────
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("Loading embedding model (all-MiniLM-L6-v2, 22M params)...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

texts = [d["content"] for d in documents]
embeddings = embedder.encode(texts, show_progress_bar=True)

# Build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings.astype(np.float32))

print(f"\n✅ Index built: {index.ntotal} vectors @ {dim} dimensions")
print(f"   Index type: Exact L2 (IndexFlatL2) — no approximation")

In [ ]:
# ─── 4. The RAG retrieval function ────────────────────────────────────────

def retrieve(query: str, top_k: int = 3) -> list[dict]:
    """Embed query, find top-K nearest documents."""
    q_vec = embedder.encode([query]).astype(np.float32)
    distances, indices = index.search(q_vec, top_k)
    
    results = []
    for i, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        doc = documents[idx]
        results.append({
            "rank": i + 1,
            "score": float(1 / (1 + dist)),  # convert distance to similarity
            "id": doc["id"],
            "title": doc["title"],
            "content": doc["content"]
        })
    return results

# Test retrieval
test_results = retrieve("how do I run AI models without internet?")
print("Query: 'how do I run AI models without internet?'")
print(f"Top {len(test_results)} results:")
for r in test_results:
    print(f"  #{r['rank']} [{r['score']:.3f}] {r['title']}")

In [ ]:
# ─── 5. The full RAG Q&A function ─────────────────────────────────────────

RAG_SYSTEM_PROMPT = """You are a helpful AI assistant. Answer questions using ONLY the provided context.
If the answer isn't in the context, say so clearly. Quote relevant parts when helpful.
Keep answers concise and factual."""

def rag_answer(question: str, top_k: int = 3, verbose: bool = True) -> str:
    """Full RAG pipeline: retrieve → prompt → generate."""
    
    # Step 1: Retrieve
    retrieved = retrieve(question, top_k=top_k)
    
    if verbose:
        print(f"📥 Retrieved {len(retrieved)} chunks:")
        for r in retrieved:
            print(f"   [{r['score']:.3f}] {r['title']}")
        print()
    
    # Step 2: Build context
    context = "\n\n".join([
        f"[Source {r['rank']}: {r['title']}]\n{r['content']}"
        for r in retrieved
    ])
    
    # Step 3: Prompt
    user_message = f"""Context:
{context}

Question: {question}"""
    
    # Step 4: Generate
    if verbose:
        print(f"🤖 {get_model()} says:")
    
    answer = chat([
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ], stream=verbose)
    
    return answer

# First RAG query
print("=" * 60)
print("Q: What makes Qwen 3.5 small models special?")
print("=" * 60)
rag_answer("What makes Qwen 3.5 small models special?")

In [ ]:
# ─── 6. More RAG queries ──────────────────────────────────────────────────

queries = [
    "How does RAG reduce hallucination?",
    "Why use tiny models instead of big ones for RAG?",
    "What is FAISS and when should I use it?",
]

for q in queries:
    print("\n" + "=" * 60)
    print(f"Q: {q}")
    print("=" * 60)
    rag_answer(q)

---
## Part 3: 👁️ Vision RAG — Images in the Pipeline

Now the fun part. Qwen 3.5 has **native vision support**. We can:

1. Send an image directly to the model for analysis  
2. Use image captions as RAG documents (caption-first approach)  
3. Do Q&A that combines retrieved text AND an image

This requires a vision-capable model:
- **Ollama**: `qwen3.5:2b` or larger (the 0.8B may have limited vision)
- **DashScope**: `qwen3.5-plus` (fully supported)

In [ ]:
# ─── 7. Vision: Direct image analysis ─────────────────────────────────────
import base64, requests
from PIL import Image
from io import BytesIO
import IPython.display as ipd

def image_to_base64(url: str) -> str:
    """Download image and convert to base64 data URI."""
    response = requests.get(url, timeout=10)
    img_data = base64.b64encode(response.content).decode("utf-8")
    content_type = response.headers.get("content-type", "image/jpeg")
    return f"data:{content_type};base64,{img_data}"

def chat_with_image(question: str, image_url: str) -> str:
    """Send a question along with an image to the vision model."""
    client = get_client()
    
    # For Ollama, use base64; for DashScope, URL works directly
    if BACKEND == "ollama":
        image_content = {"type": "image_url", "image_url": {"url": image_to_base64(image_url)}}
    else:
        image_content = {"type": "image_url", "image_url": {"url": image_url}}
    
    response = client.chat.completions.create(
        model=get_model(),
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": question},
                image_content
            ]
        }]
    )
    return response.choices[0].message.content

# Test with a public diagram image (neural network architecture diagram)
test_image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/46/Colored_neural_network.svg/296px-Colored_neural_network.svg.png"

print("Showing test image:")
ipd.display(ipd.Image(url=test_image_url, width=200))

print("\n🤖 Analyzing image with Qwen 3.5...")
description = chat_with_image(
    "Describe this image. What does it show? What are the key components?",
    test_image_url
)
print(description)

In [ ]:
# ─── 8. Vision RAG: Caption-first approach ────────────────────────────────
#
# Step 1: Generate captions for images using the vision model
# Step 2: Add captions to the RAG knowledge base
# Step 3: Retrieve via text embeddings, optionally pass original image

# Simulated image corpus (public domain images)
image_corpus = [
    {
        "id": "img_01",
        "title": "Neural Network Diagram",
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/46/Colored_neural_network.svg/296px-Colored_neural_network.svg.png",
        "caption": None  # will be generated
    },
    {
        "id": "img_02",
        "title": "Transformer Architecture",
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/8/8f/The-Transformer-model-architecture.png/300px-The-Transformer-model-architecture.png",
        "caption": None
    },
]

print("Step 1: Generating captions for image corpus...")
for img in image_corpus:
    print(f"\n  Captioning: {img['title']}")
    try:
        caption = chat_with_image(
            "Describe this technical diagram in detail for a RAG knowledge base. "
            "Include all components, labels, and their relationships.",
            img["url"]
        )
        img["caption"] = caption
        print(f"  Caption ({len(caption)} chars): {caption[:120]}...")
    except Exception as e:
        img["caption"] = f"[Caption unavailable: {e}]"
        print(f"  ⚠️  Vision error: {e}")

print("\n✅ Captioning complete")

In [ ]:
# ─── 9. Add image captions to the RAG index ───────────────────────────────

# Build image documents from captions
image_docs = [{
    "id": img["id"],
    "title": img["title"],
    "type": "image",
    "content": img["caption"],
    "source_url": img["url"]
} for img in image_corpus if img["caption"]]

# Extend the knowledge base
all_documents = documents + image_docs
all_texts = [d["content"] for d in all_documents]

# Re-embed everything
all_embeddings = embedder.encode(all_texts, show_progress_bar=False)
combined_index = faiss.IndexFlatL2(dim)
combined_index.add(all_embeddings.astype(np.float32))

print(f"✅ Extended knowledge base: {combined_index.ntotal} items")
print(f"   {len(documents)} text docs + {len(image_docs)} image captions")

def retrieve_v2(query: str, top_k: int = 3) -> list[dict]:
    """Retrieve from the combined text+image index."""
    q_vec = embedder.encode([query]).astype(np.float32)
    distances, indices = combined_index.search(q_vec, top_k)
    results = []
    for i, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        doc = all_documents[idx]
        results.append({
            "rank": i + 1,
            "score": float(1 / (1 + dist)),
            **doc
        })
    return results

# Test: does image content get retrieved?
results = retrieve_v2("show me a neural network diagram")
print("\nQuery: 'show me a neural network diagram'")
for r in results:
    icon = "🖼️" if r["type"] == "image" else "📄"
    print(f"  {icon} [{r['score']:.3f}] {r['title']}")

In [ ]:
# ─── 10. Multimodal RAG Q&A — text + image in one answer ──────────────────

def multimodal_rag_answer(question: str, top_k: int = 3) -> str:
    """RAG that can retrieve images and include them in the LLM context."""
    
    retrieved = retrieve_v2(question, top_k=top_k)
    
    print(f"📥 Retrieved {len(retrieved)} items:")
    for r in retrieved:
        icon = "🖼️" if r["type"] == "image" else "📄"
        print(f"   {icon} [{r['score']:.3f}] {r['title']}")
    print()
    
    # Build messages — text docs become context, image docs pass the actual image
    text_context = []
    image_contents = []
    
    for r in retrieved:
        if r["type"] == "text":
            text_context.append(f"[{r['title']}]\n{r['content']}")
        elif r["type"] == "image":
            text_context.append(f"[Image: {r['title']}]\nCaption: {r['content'][:200]}...")
            if BACKEND in ("ollama", "dashscope"):  # only if vision available
                if BACKEND == "ollama":
                    image_contents.append({
                        "type": "image_url",
                        "image_url": {"url": image_to_base64(r["source_url"])}
                    })
                else:
                    image_contents.append({
                        "type": "image_url",
                        "image_url": {"url": r["source_url"]}
                    })
    
    context_str = "\n\n".join(text_context)
    
    # Build the message content
    content = [{"type": "text", "text": f"{RAG_SYSTEM_PROMPT}\n\nContext:\n{context_str}\n\nQuestion: {question}"}]
    content.extend(image_contents)
    
    client = get_client()
    response = client.chat.completions.create(
        model=get_model(),
        messages=[{"role": "user", "content": content}]
    )
    
    answer = response.choices[0].message.content
    print(f"🤖 {get_model()}:\n{answer}")
    return answer

print("=" * 60)
print("Multimodal RAG Query: 'Explain how neural networks are structured'")
print("=" * 60)
multimodal_rag_answer("Explain how neural networks are structured")

---
## Part 4: Hallucination Test 🎯

Does the model stay grounded, or does it make things up when the answer isn't in the context?

In [ ]:
# ─── 11. Hallucination test ────────────────────────────────────────────────

print("Test 1: Question with answer in context")
print("=" * 60)
rag_answer("What is the context window size of Qwen 3.5 models?")

print("\n" + "=" * 60)
print("Test 2: Question NOT in context (trap question)")
print("=" * 60)
print("⚠️  Asking about something NOT in our knowledge base...")
rag_answer("What is Qwen 3.5's training data size and cutoff date?")

print("\n" + "=" * 60)
print("Test 3: Ambiguous — partially in context")
print("=" * 60)
rag_answer("Is Qwen 3.5 better than GPT-4 for all tasks?")

---
## Part 5: Benchmark — Tiny vs Larger Models

**Try swapping models** to see the quality difference on RAG tasks.

In [ ]:
# ─── 12. Model comparison (Ollama only) ───────────────────────────────────
if BACKEND == "ollama":
    import time
    
    models_to_compare = [
        ("qwen3.5:0.8b", "Qwen3.5 0.8B"),
        ("qwen3.5:2b",   "Qwen3.5 2B"),
    ]
    
    # Add 4B/9B if you have the RAM
    # models_to_compare.append(("qwen3.5:4b", "Qwen3.5 4B"))
    # models_to_compare.append(("qwen3.5:9b", "Qwen3.5 9B"))
    
    test_question = "What is the key difference between RAG approaches for vision tasks?"
    context = retrieve(test_question, top_k=2)
    context_str = "\n\n".join([f"[{r['title']}]\n{r['content']}" for r in context])
    
    client = get_client()
    results = {}
    
    for model_name, display_name in models_to_compare:
        print(f"\n{'─' * 50}")
        print(f"Testing: {display_name}")
        
        # Pull if needed
        subprocess.run(["ollama", "pull", model_name], 
                       capture_output=True, text=True)
        
        start = time.time()
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": RAG_SYSTEM_PROMPT},
                {"role": "user", "content": f"Context:\n{context_str}\n\nQ: {test_question}"}
            ]
        )
        elapsed = time.time() - start
        answer = response.choices[0].message.content
        
        results[display_name] = {"time": elapsed, "answer": answer}
        print(f"Time: {elapsed:.1f}s")
        print(f"Answer: {answer[:300]}{'...' if len(answer) > 300 else ''}")
    
    print(f"\n{'═' * 50}")
    print("Timing summary:")
    for name, r in results.items():
        print(f"  {name}: {r['time']:.1f}s")

else:
    print("Model comparison requires Ollama backend.")
    print("Switch BACKEND = 'ollama' and re-run to compare model sizes.")

---
## Part 6: Bonus — Fully Local Stack

Want everything local? Swap `sentence-transformers` embeddings for Ollama's own `nomic-embed-text` model:

In [ ]:
# ─── 13. Fully local: Ollama embeddings via nomic-embed-text ──────────────
if BACKEND == "ollama":
    
    # Pull the embedding model
    !ollama pull nomic-embed-text

    import httpx

    def ollama_embed(texts: list[str]) -> np.ndarray:
        """Get embeddings from Ollama's nomic-embed-text."""
        embeddings = []
        for text in texts:
            response = httpx.post(
                "http://localhost:11434/api/embeddings",
                json={"model": "nomic-embed-text", "prompt": text}
            )
            embeddings.append(response.json()["embedding"])
        return np.array(embeddings, dtype=np.float32)

    print("Building fully-local index with nomic-embed-text...")
    local_embeddings = ollama_embed(texts[:3])  # test with first 3 docs
    
    local_dim = local_embeddings.shape[1]
    local_index = faiss.IndexFlatL2(local_dim)
    local_index.add(local_embeddings)
    
    print(f"✅ Local index: {local_index.ntotal} vectors @ {local_dim} dims")
    print("\n🎉 Fully offline stack:")
    print(f"   Embeddings: nomic-embed-text (Ollama, {local_dim}d)")
    print(f"   Vector DB:  FAISS (in-memory)")
    print(f"   LLM:        {OLLAMA_MODEL} (Ollama)")
    print("   Network:    ❌ not needed")

else:
    print("This section requires BACKEND = 'ollama'")

---
## Summary

What we built:

```
┌──────────────────── Qwen 3.5 RAG Pipeline ────────────────────────┐
│                                                                    │
│  📚 Knowledge Base                                                 │
│     Text docs ──────┐                                             │
│     Images ──[👁️]──→ Captions ─→ Embeddings ─→ FAISS Index        │
│                                                                    │
│  ❓ Query                                                          │
│     User question ─→ Embed ─→ FAISS search ─→ Top-K chunks        │
│                                                                    │
│  💬 Generation                                                     │
│     Chunks + original images ─→ Qwen3.5 ─→ Grounded answer        │
│                                                                    │
│  📍 Where Qwen 3.5 Tiny wins:                                      │
│     • Runs offline on a laptop (2B model ~4GB RAM)                 │
│     • Native vision = no separate captioning model needed          │
│     • RAG quality gap vs GPT-4 is small — answers come from        │
│       retrieved context, not model knowledge                       │
│     • 262k context window: huge documents fit in one shot          │
└────────────────────────────────────────────────────────────────────┘
```

### Next steps
- 🔄 **Swap in your own documents** — any text files, PDFs, or image folders
- 📏 **Chunk long documents** — use `langchain.text_splitter` for large files
- 🔍 **Add hybrid search** — combine FAISS semantic + BM25 keyword for better recall
- 🏷️ **Add citation checking** — verify model answers are entailed by context
- 🤖 **Build an agent** — give the RAG system tools and let it multi-hop